# 00 · Data-Ready Pipeline

**First notebook in the Bee-A-Hero pipeline.** Runs top-to-bottom with no manual
intervention and leaves the iNaturalist dataset clean, balanced (70/15/15),
relabeled, and verified.

Stages: **inspect → validate → backup-verify → balance → relabel →
missing/duplicate/corrupted checks → directory consistency → final report.**

All heavy logic lives in `src/data_pipeline/` (`inaturalist_prep`, `label_tools`,
`eda`); this notebook only orchestrates and reports. Re-running is safe
(idempotent): the balancing step never re-deletes and never moves twice.

## Inlined source
Every `src` module below is embedded as an in-notebook module (no `from src` imports). Edit a module cell to change behaviour.

In [ ]:
import os, sys, json, glob
from pathlib import Path
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
_REPO = Path.cwd()
for _c in [_REPO, *_REPO.parents]:
    if (_c / "data").exists() and (_c / "src").exists(): _REPO = _c; break
os.chdir(_REPO)
# ---- src/config.py (inlined; edit here) ----
"""Central configuration for the Bee-A-Hero data pipeline.

All paths are anchored to the repository root (this file lives in ``src/``),
so the pipeline is portable across machines and does not depend on any
absolute/Windows path. Import this module rather than hard-coding paths.
"""

from pathlib import Path

# --- Repository layout -------------------------------------------------------
REPO_ROOT = _REPO

DATA_DIR: Path = REPO_ROOT / "data"
RAW_DIR: Path = DATA_DIR / "raw"
INTERIM_DIR: Path = DATA_DIR / "interim"
PROCESSED_DIR: Path = DATA_DIR / "processed"
BACKUP_DIR: Path = DATA_DIR / "_backup"

INAT_DIR: Path = RAW_DIR / "iNaturist"
FLOWER_DIR: Path = RAW_DIR / "Flower"

# iNaturalist splits present on disk. ``public_test`` is unlabeled
# (annotations: 0) and is inference-only — it is NOT part of the labeled split.
INAT_LABELED_SPLITS: tuple[str, ...] = ("train_mini", "val")
INAT_UNLABELED_SPLIT: str = "public_test"

# Generated-artifact locations (git-ignored; see .gitignore data/interim/*).
MANIFEST_DIR: Path = INTERIM_DIR / "manifests"
EDA_DIR: Path = INTERIM_DIR / "eda"
REMOVED_DIR: Path = BACKUP_DIR / "removed"

# --- Reproducibility ---------------------------------------------------------
SEED: int = 42

# --- Split configuration -----------------------------------------------------
# Train/val/test proportions carved from the pooled labeled Insecta images.
SPLIT_RATIOS: dict[str, float] = {"train": 0.70, "val": 0.15, "test": 0.15}

# --- Taxonomy targets --------------------------------------------------------
# Keep only folders whose taxonomic Class == Insecta.
TARGET_CLASS: str = "Insecta"

# Bee families (Hymenoptera) tagged as a subset of interest for the project.
BEE_FAMILIES: frozenset[str] = frozenset({
    "Andrenidae", "Apidae", "Colletidae", "Halictidae",
    "Megachilidae", "Melittidae", "Stenotritidae",
})

# Valid image extensions.
IMAGE_EXTS: frozenset[str] = frozenset({".jpg", ".jpeg", ".png"})
from types import SimpleNamespace
C = SimpleNamespace(**{k: v for k, v in dict(globals()).items() if k.isupper()})
REPO = _REPO
print("repo:", REPO)

#### `src/data_pipeline/eda.py` (inlined)

In [ ]:
# ===== src/data_pipeline/eda.py  (inlined module — edit here) =====
import types as _t, sys as _s
eda = _t.ModuleType("eda")
eda.C = C
_src = r'''
"""Reusable EDA & image-quality primitives for the Bee-A-Hero dataset.

These functions are imported by BOTH ``notebooks/00_data_ready.ipynb``
(integrity / quality gate) and ``notebooks/01_eda.ipynb`` (visual analysis), so
the heavy logic lives here once and the notebooks stay presentation-only.

Everything reads the Phase-4 manifest (``data/interim/manifests/split_manifest.csv``)
as the single source of truth for which images exist and their labels.
"""

import multiprocessing
import random
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import numpy as np
from PIL import Image


# Large iNaturalist frames are legitimate; disable the decompression-bomb guard.
Image.MAX_IMAGE_PIXELS = None

# Python 3.14 defaults to the "forkserver" start method, which re-imports the
# entry module in each worker and breaks under stdin/notebook execution. "fork"
# is safe here (workers only do PIL + numpy) and needs no re-import.
_MP = multiprocessing.get_context("fork")


# --------------------------------------------------------------------------- #
# Data access
# --------------------------------------------------------------------------- #
def load_manifest_df():
    """Return the Phase-4 split manifest as a DataFrame."""
    import pandas as pd
    return pd.read_csv(C.MANIFEST_DIR / "split_manifest.csv")


def manifest_paths(df=None) -> list[str]:
    df = load_manifest_df() if df is None else df
    return df["path"].tolist()


# --------------------------------------------------------------------------- #
# Integrity — corrupted / unreadable images
# --------------------------------------------------------------------------- #
def _check_one(path_str: str):
    try:
        with Image.open(path_str) as im:
            im.verify()               # verify header + payload without full decode
        return None
    except Exception as e:            # unreadable / truncated / corrupt
        return (path_str, repr(e))


def scan_corrupted(paths, workers: int | None = None, chunk: int = 64) -> list[tuple[str, str]]:
    """Return ``[(path, error)]`` for every image that fails to open/verify."""
    paths = [str(p) for p in paths]
    bad: list[tuple[str, str]] = []
    with ProcessPoolExecutor(max_workers=workers, mp_context=_MP) as ex:
        for res in ex.map(_check_one, paths, chunksize=chunk):
            if res is not None:
                bad.append(res)
    return bad


# --------------------------------------------------------------------------- #
# Per-image statistics (sampled — full set is 151k images)
# --------------------------------------------------------------------------- #
def _laplacian_var(gray: np.ndarray) -> float:
    """Variance of the Laplacian — a standard sharpness/blur proxy (numpy-only).

    Low values indicate blur (little high-frequency energy). Equivalent to the
    OpenCV ``cv2.Laplacian(...).var()`` blur score without the cv2 dependency.
    """
    g = gray
    lap = (-4.0 * g[1:-1, 1:-1]
           + g[:-2, 1:-1] + g[2:, 1:-1] + g[1:-1, :-2] + g[1:-1, 2:])
    return float(lap.var()) if lap.size else 0.0


def _meta_one(path_str: str):
    try:
        with Image.open(path_str) as im:
            w, h = im.size
            mode = im.mode
            gray = np.asarray(im.convert("L"), dtype=np.float32)
            brightness = float(gray.mean())
            contrast = float(gray.std())
            blur = _laplacian_var(gray)
            is_gray = mode in ("L", "1")
            if mode == "RGB":
                rgb = np.asarray(im, dtype=np.int16)
                if rgb.ndim == 3 and rgb.shape[2] >= 3:
                    dg = np.abs(rgb[..., 0] - rgb[..., 1]).mean()
                    db = np.abs(rgb[..., 1] - rgb[..., 2]).mean()
                    is_gray = bool(dg < 2 and db < 2)
            blank = bool(contrast < 1.0)
        return (path_str, w, h, mode, brightness, contrast, blur, int(is_gray), int(blank))
    except Exception:
        return (path_str, None, None, None, None, None, None, None, None)


def sample_image_stats(paths, sample: int | None = 6000, seed: int = C.SEED,
                       workers: int | None = None):
    """Compute width/height/aspect/mode/brightness/contrast/grayscale/blank.

    Sampled by default for speed; pass ``sample=None`` to scan everything.
    """
    import pandas as pd
    paths = [str(p) for p in paths]
    rng = random.Random(seed)
    if sample and len(paths) > sample:
        paths = rng.sample(paths, sample)
    rows = []
    with ProcessPoolExecutor(max_workers=workers, mp_context=_MP) as ex:
        for r in ex.map(_meta_one, paths, chunksize=32):
            rows.append(r)
    df = pd.DataFrame(rows, columns=["path", "width", "height", "mode",
                                     "brightness", "contrast", "blur",
                                     "is_gray", "blank"])
    df["aspect"] = df["width"] / df["height"]
    return df


# --------------------------------------------------------------------------- #
# Near-duplicate detection (perceptual hash)
# --------------------------------------------------------------------------- #
def find_near_duplicates(paths, sample: int | None = 8000, seed: int = C.SEED):
    """Group images sharing an identical perceptual hash (resize/re-encode dups).

    Returns a list of path-groups (each length >= 2). Sampled for tractability.
    """
    import imagehash
    rng = random.Random(seed)
    paths = [str(p) for p in paths]
    if sample and len(paths) > sample:
        paths = rng.sample(paths, sample)
    buckets: dict[str, list[str]] = defaultdict(list)
    for p in paths:
        try:
            with Image.open(p) as im:
                buckets[str(imagehash.phash(im))].append(p)
        except Exception:
            continue
    return [ps for ps in buckets.values() if len(ps) > 1]


# --------------------------------------------------------------------------- #
# Aggregations for plots
# --------------------------------------------------------------------------- #
def class_distribution(df=None):
    """Per-class image counts (DataFrame indexed by class_name)."""
    df = load_manifest_df() if df is None else df
    return df.groupby("class_name").size().sort_values(ascending=False)


def split_class_matrix(df=None):
    """Rows = class, columns = split, values = image counts."""
    df = load_manifest_df() if df is None else df
    return df.pivot_table(index="class_name", columns="split",
                          values="path", aggfunc="count", fill_value=0)


def order_distribution(df=None):
    """Image counts per taxonomic order."""
    df = load_manifest_df() if df is None else df
    return df.groupby("order").size().sort_values(ascending=False)


def imbalance_metrics(counts) -> dict:
    """Imbalance ratio + Gini over a series/array of per-class counts."""
    x = np.sort(np.asarray(counts, dtype=float))
    n = len(x)
    ratio = float(x.max() / x.min()) if x.min() > 0 else float("inf")
    if x.sum() == 0:
        gini = 0.0
    else:
        cum = np.cumsum(x)
        gini = float((n + 1 - 2 * (cum / cum[-1]).sum()) / n)
    return {"num_classes": n, "min": float(x.min()), "max": float(x.max()),
            "mean": float(x.mean()), "imbalance_ratio": ratio, "gini": round(gini, 4)}


def sample_grid_paths(df=None, per_split: int = 8, seed: int = C.SEED) -> list[str]:
    """Deterministic sample of image paths for a preview grid."""
    df = load_manifest_df() if df is None else df
    rng = random.Random(seed)
    out: list[str] = []
    for split in ("train", "val", "test"):
        pool = df[df["split"] == split]["path"].tolist()
        out += rng.sample(pool, min(per_split, len(pool)))
    return out
'''
exec(compile(_src, "src/data_pipeline/eda.py", "exec"), eda.__dict__)
_s.modules["eda"] = eda
globals()["eda"] = eda

#### `src/data_pipeline/inaturalist_prep.py` (inlined)

In [ ]:
# ===== src/data_pipeline/inaturalist_prep.py  (inlined module — edit here) =====
import types as _t, sys as _s
prep = _t.ModuleType("prep")
prep.C = C
_src = r'''
"""iNaturalist dataset preparation: filter → dedup → 70/15/15 split.

Design goals (locked with the team):
  * **Non-destructive.** Nothing is ``rm``-ed. "Unnecessary" images (non-Insecta
    folders, exact duplicates) are *moved* to ``data/_backup/removed/`` mirroring
    their original relative path, and every moved image is logged. Fully
    reversible.
  * **Logical split.** The train/val/test split is a *manifest* (which image
    belongs to which split); images are NOT physically shuffled between split
    folders, so there is no duplication and no cross-split leakage by move.
  * **Reproducible.** Fixed seed, deterministic ordering, hash cache. Running
    twice yields the same manifest and is idempotent on disk.

Pool = Insecta images from ``train_mini`` + ``val`` (public_test is unlabeled →
excluded). The dataset is uniform (60 img/class), so 70/15/15 = 42/9/9 per class
with no truncation; the only removals are non-Insecta and exact duplicates.

CLI
    python -m src.data_pipeline.inaturalist_prep            # dry-run (plan only)
    python -m src.data_pipeline.inaturalist_prep --apply    # execute moves
    python -m src.data_pipeline.inaturalist_prep --apply --skip-dedup
"""

import argparse
import csv
import hashlib
import json
import os
import random
import shutil
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path


# --------------------------------------------------------------------------- #
# Taxonomy parsing
# --------------------------------------------------------------------------- #
# Folder name: ID_Kingdom_Phylum_Class_Order_Family_Genus_specificEpithet
_TAXON_FIELDS = ("tax_id", "kingdom", "phylum", "class", "order",
                 "family", "genus", "species")


def parse_taxon(folder_name: str) -> dict[str, str]:
    """Split an iNaturalist species-folder name into taxonomy fields.

    Names have exactly 8 underscore-separated parts. If a species epithet
    itself contains underscores the extras are re-joined into ``species``.
    """
    parts = folder_name.split("_")
    if len(parts) < len(_TAXON_FIELDS):
        return {f: "" for f in _TAXON_FIELDS} | {"tax_id": folder_name}
    head = parts[: len(_TAXON_FIELDS) - 1]
    species = "_".join(parts[len(_TAXON_FIELDS) - 1:])
    values = head + [species]
    return dict(zip(_TAXON_FIELDS, values))


def is_insecta(taxon: dict[str, str]) -> bool:
    return taxon.get("class", "") == C.TARGET_CLASS


def is_bee(taxon: dict[str, str]) -> bool:
    return taxon.get("order", "") == "Hymenoptera" and taxon.get("family", "") in C.BEE_FAMILIES


# --------------------------------------------------------------------------- #
# Scanning
# --------------------------------------------------------------------------- #
def iter_class_dirs(split_dir: Path):
    """Yield species sub-folders of a split directory, sorted for determinism."""
    if not split_dir.is_dir():
        return
    for d in sorted(p for p in split_dir.iterdir() if p.is_dir()):
        yield d


def _images_in(folder: Path) -> list[Path]:
    return sorted(p for p in folder.iterdir()
                  if p.is_file() and p.suffix.lower() in C.IMAGE_EXTS)


def _rel(path: Path) -> str:
    return str(path.relative_to(C.REPO_ROOT))


def _now() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


# --------------------------------------------------------------------------- #
# Dataset availability (the raw images are git-ignored — a fresh clone is empty)
# --------------------------------------------------------------------------- #
DATASET_HELP = (
    "iNaturalist raw data not found under data/raw/iNaturist/.\n"
    "The image dataset is git-ignored (too large to version), so a fresh clone\n"
    "has an empty data/ tree. To run the pipeline, obtain the iNaturalist 2021\n"
    "archives and extract them so the layout is:\n"
    "    data/raw/iNaturist/train_mini/   (+ train_mini.json)\n"
    "    data/raw/iNaturist/val/          (+ val.json)\n"
    "    data/raw/iNaturist/public_test/  (+ public_test.json)\n"
    "Source: https://github.com/visipedia/inat_comp/tree/master/2021\n"
    "Then re-run — the seeded pipeline reproduces the exact same splits/labels."
)


def dataset_present() -> bool:
    """True if at least one labeled split has species sub-folders on disk."""
    for split in C.INAT_LABELED_SPLITS:
        d = C.INAT_DIR / split
        if d.is_dir() and any(p.is_dir() for p in d.iterdir()):
            return True
    return False


def require_dataset() -> None:
    """Raise a clear, actionable error if the raw dataset is missing."""
    if not dataset_present():
        raise FileNotFoundError(DATASET_HELP)


# --------------------------------------------------------------------------- #
# Backup move (the only mutating primitive)
# --------------------------------------------------------------------------- #
def move_to_backup(path: Path, reason: str, apply: bool) -> dict:
    """Move ``path`` under REMOVED_DIR mirroring its repo-relative location.

    Returns a log row. In dry-run (``apply=False``) nothing is moved.
    """
    rel = path.relative_to(C.REPO_ROOT)
    dest = C.REMOVED_DIR / rel
    if apply:
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(path), str(dest))
    return {
        "original_path": str(rel),
        "backup_path": str(dest.relative_to(C.REPO_ROOT)),
        "reason": reason,
        "timestamp": _now(),
    }


# --------------------------------------------------------------------------- #
# Stage 1 — Insecta filter
# --------------------------------------------------------------------------- #
def filter_non_insecta(apply: bool) -> list[dict]:
    """Move every non-Insecta species folder (all its images) to backup.

    Returns per-image removal log rows.
    """
    removed: list[dict] = []
    for split in C.INAT_LABELED_SPLITS:
        for folder in list(iter_class_dirs(C.INAT_DIR / split)):
            taxon = parse_taxon(folder.name)
            if is_insecta(taxon):
                continue
            imgs = _images_in(folder)
            for img in imgs:
                row = move_to_backup(img, reason="non_insecta", apply=apply)
                row["class_name"] = folder.name
                row["split"] = split
                removed.append(row)
            # remove the now-empty (or fully-moved) folder shell on apply
            if apply and folder.exists() and not any(folder.iterdir()):
                folder.rmdir()
    return removed


# --------------------------------------------------------------------------- #
# Stage 2 — build the Insecta image pool
# --------------------------------------------------------------------------- #
def build_pool() -> list[dict]:
    """Return one record per Insecta image across labeled splits.

    Class folders exist identically in both splits, so a class is keyed by its
    folder name; its images are pooled from train_mini + val.
    """
    records: list[dict] = []
    for split in C.INAT_LABELED_SPLITS:
        for folder in iter_class_dirs(C.INAT_DIR / split):
            taxon = parse_taxon(folder.name)
            if not is_insecta(taxon):
                continue
            bee = is_bee(taxon)
            for img in _images_in(folder):
                records.append({
                    "path": _rel(img),
                    "class_name": folder.name,
                    "order": taxon["order"],
                    "family": taxon["family"],
                    "genus": taxon["genus"],
                    "species": taxon["species"],
                    "is_bee": int(bee),
                    "source_split": split,
                })
    return records


# --------------------------------------------------------------------------- #
# Stage 3 — exact duplicate removal (md5, cached)
# --------------------------------------------------------------------------- #
def _md5(path: Path, chunk: int = 1 << 20) -> str:
    h = hashlib.md5()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def _load_hash_cache(cache_path: Path) -> dict[str, tuple[int, int, str]]:
    cache: dict[str, tuple[int, int, str]] = {}
    if cache_path.exists():
        with open(cache_path, newline="") as fh:
            for r in csv.DictReader(fh):
                cache[r["path"]] = (int(r["size"]), int(r["mtime"]), r["md5"])
    return cache


def _save_hash_cache(cache_path: Path, cache: dict[str, tuple[int, int, str]]) -> None:
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    with open(cache_path, "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["path", "size", "mtime", "md5"])
        for p, (size, mtime, md5) in sorted(cache.items()):
            w.writerow([p, size, mtime, md5])


def dedup_exact(records: list[dict], apply: bool) -> tuple[list[dict], list[dict]]:
    """Remove exact-duplicate images (identical md5), keeping the first by path.

    Returns ``(kept_records, removed_log_rows)``. Uses a size+mtime-keyed cache
    so re-runs don't re-hash unchanged files.
    """
    cache_path = C.MANIFEST_DIR / "md5_cache.csv"
    cache = _load_hash_cache(cache_path)
    updated = dict(cache)

    for rec in records:
        p = C.REPO_ROOT / rec["path"]
        st = p.stat()
        cached = cache.get(rec["path"])
        if cached and cached[0] == st.st_size and cached[1] == int(st.st_mtime):
            md5 = cached[2]
        else:
            md5 = _md5(p)
            updated[rec["path"]] = (st.st_size, int(st.st_mtime), md5)
        rec["md5"] = md5
    _save_hash_cache(cache_path, updated)

    seen: dict[str, str] = {}          # md5 -> first path kept
    kept: list[dict] = []
    removed: list[dict] = []
    for rec in sorted(records, key=lambda r: r["path"]):
        md5 = rec["md5"]
        if md5 in seen:
            row = move_to_backup(C.REPO_ROOT / rec["path"],
                                 reason="exact_duplicate", apply=apply)
            row["class_name"] = rec["class_name"]
            row["split"] = rec["source_split"]
            row["md5"] = md5
            row["duplicate_of"] = seen[md5]
            removed.append(row)
        else:
            seen[md5] = rec["path"]
            kept.append(rec)
    return kept, removed


# --------------------------------------------------------------------------- #
# Stage 4 — stratified 70/15/15 split (largest remainder, no data loss)
# --------------------------------------------------------------------------- #
def _largest_remainder(n: int, ratios: dict[str, float]) -> dict[str, int]:
    """Split ``n`` items into named buckets summing exactly to ``n``."""
    raw = {k: n * v for k, v in ratios.items()}
    base = {k: int(v) for k, v in raw.items()}
    remaining = n - sum(base.values())
    # distribute leftovers to the largest fractional parts (stable order)
    order = sorted(ratios, key=lambda k: (-(raw[k] - base[k]), k))
    for i in range(remaining):
        base[order[i % len(order)]] += 1
    return base


def assign_splits(kept: list[dict], seed: int) -> list[dict]:
    """Assign each kept image a train/val/test split, stratified per class."""
    by_class: dict[str, list[dict]] = defaultdict(list)
    for rec in kept:
        by_class[rec["class_name"]].append(rec)

    class_names = sorted(by_class)
    class_to_idx = {name: i for i, name in enumerate(class_names)}
    rng = random.Random(seed)

    for name in class_names:
        recs = sorted(by_class[name], key=lambda r: r["path"])
        rng.shuffle(recs)
        alloc = _largest_remainder(len(recs), C.SPLIT_RATIOS)
        i = 0
        for split in ("train", "val", "test"):
            for rec in recs[i:i + alloc[split]]:
                rec["split"] = split
                rec["class_id"] = class_to_idx[name]
                rec["status"] = "kept"
            i += alloc[split]
    return class_names


# --------------------------------------------------------------------------- #
# Stage 5 — downsample the (unlabeled) public_test folder
# --------------------------------------------------------------------------- #
def reduce_public_test(target: int, apply: bool) -> dict:
    """Shrink the huge unlabeled ``public_test/`` folder to ``target`` images.

    Deterministic (seeded) so every teammate keeps the *same* subset: the flat
    image list is sorted, shuffled with ``config.SEED``, and the first
    ``target`` are kept. Surplus images are moved to ``data/_backup/removed/``
    (never deleted here). Idempotent — a no-op once the folder is already at or
    below ``target``. Writes the reproducible kept-list to
    ``manifests/public_test_kept.txt``.
    """
    pt = C.INAT_DIR / C.INAT_UNLABELED_SPLIT
    result = {"target": target, "action": "none"}
    if not pt.is_dir():
        result["action"] = "skip (no public_test dir)"
        return result

    imgs = sorted(p for p in pt.iterdir()
                  if p.is_file() and p.suffix.lower() in C.IMAGE_EXTS)
    total = len(imgs)
    result["public_test_total"] = total

    if total <= target:
        keep = imgs
        surplus: list[Path] = []
        result["action"] = "skip (already <= target)"
    else:
        order = list(imgs)                      # already sorted -> deterministic
        random.Random(C.SEED).shuffle(order)
        keep_set = set(order[:target])
        keep = [p for p in imgs if p in keep_set]
        surplus = [p for p in imgs if p not in keep_set]
        result["action"] = "APPLIED" if apply else "DRY_RUN"

    C.MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
    (C.MANIFEST_DIR / "public_test_kept.txt").write_text(
        "\n".join(_rel(p) for p in sorted(keep)) + "\n")

    if apply and surplus:
        dest_dir = C.REMOVED_DIR / pt.relative_to(C.REPO_ROOT)
        dest_dir.mkdir(parents=True, exist_ok=True)
        for p in surplus:
            os.rename(p, dest_dir / p.name)      # same filesystem -> fast rename

    result["kept"] = len(keep)
    result["removed"] = len(surplus)
    return result


# --------------------------------------------------------------------------- #
# Writers
# --------------------------------------------------------------------------- #
_MANIFEST_COLS = ["path", "class_id", "class_name", "order", "family", "genus",
                  "species", "is_bee", "source_split", "split", "status"]
_REMOVED_COLS = ["original_path", "backup_path", "reason", "class_name",
                 "split", "md5", "duplicate_of", "timestamp"]


def _write_csv(path: Path, cols: list[str], rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=cols, extrasaction="ignore")
        w.writeheader()
        for r in rows:
            w.writerow(r)


def write_outputs(kept: list[dict], removed: list[dict], class_names: list[str]) -> dict:
    C.MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
    manifest = sorted(kept, key=lambda r: (r["class_id"], r["split"], r["path"]))
    _write_csv(C.MANIFEST_DIR / "split_manifest.csv", _MANIFEST_COLS, manifest)
    # Preserve the audit log on idempotent re-runs: only (re)write when this run
    # actually removed something, or when no log exists yet. Otherwise a second
    # --apply (which finds nothing left to remove) would wipe the record.
    removed_log = C.MANIFEST_DIR / "removed_log.csv"
    if removed or not removed_log.exists():
        _write_csv(removed_log, _REMOVED_COLS, removed)

    class_to_idx = {name: i for i, name in enumerate(class_names)}
    with open(C.MANIFEST_DIR / "class_index.json", "w") as fh:
        json.dump({"class_to_idx": class_to_idx,
                   "idx_to_class": {i: n for n, i in class_to_idx.items()},
                   "num_classes": len(class_names)}, fh, indent=2)

    counts = {"train": 0, "val": 0, "test": 0}
    bee = {"train": 0, "val": 0, "test": 0}
    for r in kept:
        counts[r["split"]] += 1
        bee[r["split"]] += r["is_bee"]
    reasons: dict[str, int] = defaultdict(int)
    for r in removed:
        reasons[r["reason"]] += 1

    summary = {
        "generated": _now(),
        "seed": C.SEED,
        "ratios": C.SPLIT_RATIOS,
        "num_classes": len(class_names),
        "kept_images": len(kept),
        "split_counts": counts,
        "split_fractions": {k: round(v / max(len(kept), 1), 4) for k, v in counts.items()},
        "bee_images_per_split": bee,
        "removed_total": len(removed),
        "removed_by_reason": dict(reasons),
    }
    with open(C.MANIFEST_DIR / "split_summary.json", "w") as fh:
        json.dump(summary, fh, indent=2)
    return summary


# --------------------------------------------------------------------------- #
# Orchestration
# --------------------------------------------------------------------------- #
def run(apply: bool = False, skip_dedup: bool = False) -> dict:
    require_dataset()                       # clear error if raw data is missing
    removed: list[dict] = []
    removed += filter_non_insecta(apply=apply)

    records = build_pool()
    if skip_dedup:
        kept = records
        for r in kept:
            r["md5"] = ""
    else:
        kept, dup_removed = dedup_exact(records, apply=apply)
        removed += dup_removed

    class_names = assign_splits(kept, seed=C.SEED)
    summary = write_outputs(kept, removed, class_names)

    # Stage 5: shrink the oversized unlabeled public_test folder to match the
    # val split size (seeded -> reproducible across teammates).
    val_count = summary["split_counts"]["val"]
    summary["public_test_reduction"] = reduce_public_test(val_count, apply=apply)

    summary["mode"] = "APPLIED" if apply else "DRY_RUN"
    with open(C.MANIFEST_DIR / "split_summary.json", "w") as fh:
        json.dump(summary, fh, indent=2)
    return summary


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--apply", action="store_true",
                    help="execute moves (default: dry-run, manifests only)")
    ap.add_argument("--skip-dedup", action="store_true",
                    help="skip exact-duplicate md5 scan")
    args = ap.parse_args()
    summary = run(apply=args.apply, skip_dedup=args.skip_dedup)
    print(json.dumps(summary, indent=2))


if __name__ == "__main__":
    main()
'''
exec(compile(_src, "src/data_pipeline/inaturalist_prep.py", "exec"), prep.__dict__)
_s.modules["prep"] = prep
globals()["prep"] = prep

#### `src/data_pipeline/label_tools.py` (inlined)

In [ ]:
# ===== src/data_pipeline/label_tools.py  (inlined module — edit here) =====
import types as _t, sys as _s
labels = _t.ModuleType("labels")
labels.C = C
_src = r'''
"""Label regeneration & integrity validation for the cleaned iNaturalist set.

After Phase-4 filtering the original iNaturalist COCO JSONs
(``train_mini.json`` 500k refs, ``val.json`` 100k refs, 10k categories) no
longer match what is on disk (2526 Insecta classes, 151,545 images). This
module rebuilds labels so they reference *only surviving images* and validates
that no orphan/broken reference remains.

It NEVER touches the original raw JSONs (those are the backup in
``data/_backup/original_labels/``). All output goes to
``data/interim/labels/``:

  * ``inat_<split>_filtered.json`` — the original COCO file with every image /
    annotation / category that no longer exists removed.
  * ``{train,val,test}.json``     — clean COCO-classification labels built from
    the Phase-4 manifest (contiguous category ids = training class_id).
  * ``validation_report.json``    — the Phase-5 integrity checklist result.

CLI
    python -m src.data_pipeline.label_tools           # build + validate
    python -m src.data_pipeline.label_tools --validate-only
"""

import argparse
import csv
import json
from collections import Counter, defaultdict
from pathlib import Path


LABELS_DIR: Path = C.INTERIM_DIR / "labels"
_INAT_PREFIX = "data/raw/iNaturist/"


# --------------------------------------------------------------------------- #
# Helpers
# --------------------------------------------------------------------------- #
def _load_manifest() -> list[dict]:
    with open(C.MANIFEST_DIR / "split_manifest.csv", newline="") as fh:
        return list(csv.DictReader(fh))


def _json_name(manifest_path: str) -> str:
    """Manifest path -> the file_name used inside the original iNat JSONs."""
    return manifest_path[len(_INAT_PREFIX):] if manifest_path.startswith(_INAT_PREFIX) else manifest_path


# --------------------------------------------------------------------------- #
# Filter the original COCO JSONs down to surviving images
# --------------------------------------------------------------------------- #
def filter_original(split_json: str, survivor_names: set[str]) -> tuple[dict, dict]:
    """Drop images/annotations/categories that no longer exist on disk.

    Returns ``(filtered_coco, stats)``.
    """
    src = C.INAT_DIR / split_json
    coco = json.load(open(src))
    imgs = [im for im in coco["images"] if im["file_name"] in survivor_names]
    kept_ids = {im["id"] for im in imgs}
    anns = [a for a in coco["annotations"] if a["image_id"] in kept_ids]
    used_cat = {a["category_id"] for a in anns}
    cats = [c for c in coco["categories"] if c["id"] in used_cat]
    filtered = {k: coco.get(k) for k in ("info", "licenses") if k in coco}
    filtered |= {"images": imgs, "annotations": anns, "categories": cats}
    stats = {
        "source": split_json,
        "images_before": len(coco["images"]), "images_after": len(imgs),
        "annotations_before": len(coco["annotations"]), "annotations_after": len(anns),
        "categories_before": len(coco["categories"]), "categories_after": len(cats),
    }
    return filtered, stats


def _dims_and_origcat(filtered_by_split: dict[str, dict]) -> tuple[dict, dict]:
    """Index file_name -> (width, height) and dir_name -> original category obj."""
    dims: dict[str, tuple[int, int]] = {}
    cat_by_dir: dict[str, dict] = {}
    for coco in filtered_by_split.values():
        for im in coco["images"]:
            dims[im["file_name"]] = (im.get("width"), im.get("height"))
        for c in coco["categories"]:
            cat_by_dir.setdefault(c.get("image_dir_name", c.get("name")), c)
    return dims, cat_by_dir


# --------------------------------------------------------------------------- #
# Build clean per-split labels from the manifest
# --------------------------------------------------------------------------- #
def build_clean_labels(manifest: list[dict], dims: dict, cat_by_dir: dict) -> dict:
    """Write clean COCO-classification JSON for train/val/test. Returns stats."""
    # category catalog: contiguous training id -> taxonomy (+ original iNat id)
    classes: dict[int, dict] = {}
    for r in manifest:
        cid = int(r["class_id"])
        if cid in classes:
            continue
        orig = cat_by_dir.get(r["class_name"], {})
        classes[cid] = {
            "id": cid,
            "name": r["class_name"],
            "order": r["order"], "family": r["family"],
            "genus": r["genus"], "species": r["species"],
            "is_bee": int(r["is_bee"]),
            "inat_category_id": orig.get("id"),
            "common_name": orig.get("common_name"),
        }
    categories = [classes[i] for i in sorted(classes)]

    stats = {}
    for split in ("train", "val", "test"):
        rows = [r for r in manifest if r["split"] == split]
        images, annotations = [], []
        for i, r in enumerate(sorted(rows, key=lambda x: x["path"])):
            jn = _json_name(r["path"])
            w, h = dims.get(jn, (None, None))
            images.append({"id": i, "file_name": r["path"], "width": w, "height": h})
            annotations.append({"id": i, "image_id": i, "category_id": int(r["class_id"])})
        coco = {"info": {"description": f"Bee-A-Hero iNaturalist Insecta — {split}"},
                "images": images, "annotations": annotations, "categories": categories}
        LABELS_DIR.mkdir(parents=True, exist_ok=True)
        json.dump(coco, open(LABELS_DIR / f"{split}.json", "w"))
        stats[split] = {"images": len(images), "annotations": len(annotations)}
    stats["num_categories"] = len(categories)
    return stats


# --------------------------------------------------------------------------- #
# Phase-5 integrity checklist
# --------------------------------------------------------------------------- #
def validate(manifest: list[dict]) -> dict:
    nc = len({int(r["class_id"]) for r in manifest})
    manifest_paths = [r["path"] for r in manifest]
    manifest_set = set(manifest_paths)

    # images on disk (labeled splits) vs manifest
    disk = {str(p.relative_to(C.REPO_ROOT))
            for s in C.INAT_LABELED_SPLITS
            for p in (C.INAT_DIR / s).rglob("*")
            if p.is_file() and p.suffix.lower() in C.IMAGE_EXTS}

    images_without_labels = sorted(disk - manifest_set)          # on disk, no manifest row
    labels_without_images = sorted(manifest_set - disk)          # manifest row, file gone
    dup_counts = Counter(manifest_paths)
    duplicated_labels = sorted(p for p, n in dup_counts.items() if n > 1)
    invalid_category = sorted(r["path"] for r in manifest
                              if not (0 <= int(r["class_id"]) < nc))
    corrupted = sorted(r["path"] for r in manifest
                       if not r["class_name"] or r["split"] not in ("train", "val", "test"))
    # cross-split leakage: a path assigned to >1 split
    split_of = defaultdict(set)
    for r in manifest:
        split_of[r["path"]].add(r["split"])
    leakage = sorted(p for p, s in split_of.items() if len(s) > 1)

    report = {
        "num_classes": nc,
        "manifest_images": len(manifest),
        "disk_images": len(disk),
        "images_without_labels": len(images_without_labels),
        "labels_without_images": len(labels_without_images),
        "duplicated_labels": len(duplicated_labels),
        "invalid_category_ids": len(invalid_category),
        "corrupted_labels": len(corrupted),
        "cross_split_leakage": len(leakage),
        "examples": {
            "images_without_labels": images_without_labels[:5],
            "labels_without_images": labels_without_images[:5],
        },
    }
    report["all_checks_pass"] = all(report[k] == 0 for k in (
        "images_without_labels", "labels_without_images", "duplicated_labels",
        "invalid_category_ids", "corrupted_labels", "cross_split_leakage"))
    return report


# --------------------------------------------------------------------------- #
# Orchestration
# --------------------------------------------------------------------------- #
def run(validate_only: bool = False) -> dict:
    manifest = _load_manifest()
    result: dict = {}

    if not validate_only:
        # survivor file_names per source split, then filter the originals
        survivors_by_split: dict[str, set[str]] = defaultdict(set)
        for r in manifest:
            survivors_by_split[r["source_split"]].add(_json_name(r["path"]))

        LABELS_DIR.mkdir(parents=True, exist_ok=True)
        filtered_by_split: dict[str, dict] = {}
        filter_stats = []
        for split in C.INAT_LABELED_SPLITS:
            filtered, stats = filter_original(f"{split}.json", survivors_by_split[split])
            json.dump(filtered, open(LABELS_DIR / f"inat_{split}_filtered.json", "w"))
            filtered_by_split[split] = filtered
            filter_stats.append(stats)
        dims, cat_by_dir = _dims_and_origcat(filtered_by_split)
        result["filter_original"] = filter_stats
        result["clean_labels"] = build_clean_labels(manifest, dims, cat_by_dir)

    result["validation"] = validate(manifest)
    json.dump(result, open(C.MANIFEST_DIR / "validation_report.json", "w"), indent=2)
    return result


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--validate-only", action="store_true")
    args = ap.parse_args()
    print(json.dumps(run(validate_only=args.validate_only), indent=2))


if __name__ == "__main__":
    main()
'''
exec(compile(_src, "src/data_pipeline/label_tools.py", "exec"), labels.__dict__)
_s.modules["labels"] = labels
globals()["labels"] = labels

### Dataset availability
The raw images are **git-ignored** (too large to version), so a fresh clone has an empty `data/` tree. This cell checks the dataset is present and, if not, prints exactly how to obtain it.

In [ ]:
if not prep.dataset_present():
    print(prep.DATASET_HELP)
    raise SystemExit("iNaturalist dataset not found — follow the instructions above, then re-run.")
print("iNaturalist dataset found — proceeding.")

## 1 · Dataset inspection

In [ ]:
def inspect():
    info = {}
    for split in C.INAT_LABELED_SPLITS:
        d = C.INAT_DIR / split
        if not d.is_dir():
            info[split] = {"missing": True}; continue
        folders = [f for f in d.iterdir() if f.is_dir()]
        insecta = [f for f in folders if "_Insecta_" in f.name]
        imgs = sum(1 for _ in d.rglob("*") if _.is_file() and _.suffix.lower() in C.IMAGE_EXTS)
        info[split] = {"folders": len(folders), "insecta_folders": len(insecta),
                       "non_insecta_folders": len(folders) - len(insecta), "images": imgs}
    pt = C.INAT_DIR / C.INAT_UNLABELED_SPLIT
    info[C.INAT_UNLABELED_SPLIT] = {"images": sum(1 for _ in pt.glob('*') if _.is_file()),
                                    "note": "unlabeled — inference only, not in split"}
    return info

REPORT["inspection"] = inspect()
print(json.dumps(REPORT["inspection"], indent=2))

## 2 · Backup verification
If the Phase-3 safety backup is present, verify the original labels are intact (md5). The backup is **optional** — once the dataset is finalized it may be removed to save space, so a missing backup is not a failure.

In [ ]:
import hashlib
bk = C.BACKUP_DIR
if not bk.exists():
    REPORT["backup_verification"] = {"present": False, "ok": True,
        "note": "backup removed post-cleanup — not required"}
else:
    md5_ok = True
    sums = bk / "original_labels" / "MD5SUMS.txt"
    if sums.exists():
        for line in sums.read_text().splitlines():
            want, name = line.split()
            p = bk / "original_labels" / name
            got = hashlib.md5(p.read_bytes()).hexdigest() if p.exists() else None
            md5_ok &= (got == want)
    REPORT["backup_verification"] = {"present": True,
        "original_labels_md5_ok": bool(md5_ok and sums.exists()),
        "ok": bool(md5_ok and sums.exists())}
print(json.dumps(REPORT["backup_verification"], indent=2))

## 3 · Dataset balancing (Insecta filter · exact dedup · 70/15/15 · public_test downsample)
Idempotent and seeded: non-Insecta + exact duplicates are *moved* to `data/_backup/removed/` (never deleted) and logged. The oversized unlabeled `public_test/` folder is downsampled (seeded) to the **val split size** so teammates keep the identical subset.

In [ ]:
REPORT["balancing"] = prep.run(apply=True)     # safe to re-run
print(json.dumps(REPORT["balancing"], indent=2))

## 4 · Label regeneration / update
Rebuild labels to reference only surviving images; filter the original iNat JSONs and emit clean per-split COCO labels.

In [ ]:
res = labels.run()
REPORT["label_regeneration"] = {"filter_original": res["filter_original"],
                                "clean_labels": res["clean_labels"]}
REPORT["label_validation"] = res["validation"]
print(json.dumps(REPORT["label_validation"], indent=2))

## 5 · Integrity checks
Missing files · duplicate labels · corrupted images · near-duplicates · directory consistency.

In [ ]:
df = eda.load_manifest_df()
paths = df["path"].tolist()

# 5a — missing files (manifest rows whose image is gone)
missing = [p for p in paths if not (REPO / p).exists()]

# 5b — duplicate labels (same image path listed twice)
dups = df["path"].value_counts()
duplicate_labels = dups[dups > 1].index.tolist()

# 5c — corrupted / unreadable images (FULL scan, all splits)
print(f"Scanning {len(paths):,} images for corruption ...")
corrupted = eda.scan_corrupted(paths, workers=os.cpu_count())

# 5d — perceptual near-duplicates (sampled)
near_dup_groups = eda.find_near_duplicates(paths, sample=8000)

REPORT["integrity"] = {
    "missing_files": len(missing),
    "duplicate_labels": len(duplicate_labels),
    "corrupted_images": len(corrupted),
    "near_duplicate_groups_sampled": len(near_dup_groups),
    "examples": {"missing": missing[:5],
                 "corrupted": [c[0] for c in corrupted[:5]]},
}
print(json.dumps(REPORT["integrity"], indent=2))

### 5e · Directory consistency

In [ ]:
train_classes = {f.name for f in (C.INAT_DIR / "train_mini").iterdir() if f.is_dir()}
val_classes   = {f.name for f in (C.INAT_DIR / "val").iterdir() if f.is_dir()}
dir_checks = {
    "train_folders": len(train_classes),
    "val_folders": len(val_classes),
    "class_sets_identical": train_classes == val_classes,
    "manifest_classes": int(df["class_id"].nunique()),
    "all_paths_under_labeled_splits":
        bool(df["path"].str.startswith("data/raw/iNaturist/").all()),
    "every_class_has_all_splits":
        bool((df.pivot_table(index="class_name", columns="split",
              values="path", aggfunc="count", fill_value=0) > 0).all().all()),
}
REPORT["directory_consistency"] = dir_checks
print(json.dumps(dir_checks, indent=2))

## 6 · Final report
Aggregate every check into a single PASS/FAIL gate and persist it. The notebook **fails loudly** (assert) if the dataset is not ready.

In [ ]:
integ = REPORT["integrity"]; val = REPORT["label_validation"]; dc = REPORT["directory_consistency"]
gate = {
    "backup_verified": REPORT["backup_verification"]["ok"],
    "labels_valid": val["all_checks_pass"],
    "no_missing_files": integ["missing_files"] == 0,
    "no_duplicate_labels": integ["duplicate_labels"] == 0,
    "no_corrupted_images": integ["corrupted_images"] == 0,
    "class_sets_identical": dc["class_sets_identical"],
    "every_class_all_splits": dc["every_class_has_all_splits"],
    "split_is_70_15_15": REPORT["balancing"]["split_fractions"] ==
                         {"train": 0.7, "val": 0.15, "test": 0.15},
    "public_test_reduced_to_val":
        REPORT["balancing"].get("public_test_reduction", {}).get("kept")
        == REPORT["balancing"]["split_counts"]["val"],
}
REPORT["gate"] = gate
REPORT["DATA_READY"] = all(gate.values())

out_dir = C.INTERIM_DIR / "reports"; out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / "data_ready_report.json").write_text(json.dumps(REPORT, indent=2))

print(json.dumps(gate, indent=2))
print("\n=== DATA_READY:", REPORT["DATA_READY"], "===")
print("report ->", out_dir / "data_ready_report.json")
assert REPORT["DATA_READY"], "Dataset is NOT ready — see failing gate checks above."